V1 (02 11 2025): Translated from R to Python and added the visualizations for goals B to D


V2 (04 01 2025): Loop system implemented

V2.5 (04 02 2025): Loops system tweaks. Removed openpyxl as it was corrupting files and replaced it with xlwings

V3 & V4 finalize the loop except for the tax and asset channel

In [28]:
#!pip install numpy pandas scipy matplotlib xlwings

In [29]:
import numpy as np
from scipy.stats import norm
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import os
import subprocess
import xlwings as xw
import time


In [30]:
# -- Variable Setup -- ##

#Monte Carlo Trials
n_trials = 10**5

#Case Study Profile Selection
Profile = "P1" #Either P1 or P2

#Excel worksheets
excel_returns = "Returns3"
excel_volatilities = "Volatilities"
excel_correlation = "Correlation"
excel_gbi = "GBI Allocations P1" if Profile == "P1" else "GBI Allocations P2"
excel_gbi_goals = "GBI Goals P1" if Profile == "P1" else "GBI Goals P2"


# Define Functions
This section defines the functions used for calculating portfolio volatility, expected return, 
goal achievement probability, and the objective (failure probability) to minimize.

In [31]:

def sd_f(weight_vector, covar_table):
    covar_vector = np.zeros(len(weight_vector))
    for z in range(len(weight_vector)):
        covar_vector[z] = np.sum(weight_vector * covar_table[:, z])
    return np.sqrt(np.sum(weight_vector * covar_vector))

In [32]:
def mean_f(weight_vector, return_vector):
    return np.sum(weight_vector * return_vector)

In [33]:
"""def phi_f(goal_vector, goal_allocation, pool, mean, sd):
    # goal_vector is [value ratio, funding requirement, time horizon]
    required_return = (goal_vector[1] / (pool * goal_allocation))**(1 / goal_vector[2]) - 1
    if goal_allocation * pool >= goal_vector[1]:
        return 1
    else:
        return 1 - norm.cdf(required_return, loc=mean, scale=sd)

""" #Made new version below that takes into account cases where the goal is too unreachable and writes down when that happens

def phi_f(goal_vector, goal_allocation, pool, mean, sd, year=None, goal_name=None):
    denominator = pool * goal_allocation
    if denominator <= 0:
        if year and goal_name:
            infeasible_goals_log.append((year, goal_name, "Zero allocation or pool"))
        return 0

    required_return = (goal_vector[1] / denominator)**(1 / goal_vector[2]) - 1

    if required_return > 1:
        if year and goal_name:
            infeasible_goals_log.append((year, goal_name, f"Required return > 100% ({required_return:.2%})"))
        return 0

    if denominator >= goal_vector[1]:
        return 1

    return 1 - norm.cdf(required_return, loc=mean, scale=sd)



In [34]:
def optim_function(weights):
    # Uses the current global variables: goal_vector, allocation, pool, return_vector, covar_table
    return 1 - phi_f(
        goal_vector,
        allocation,
        pool,
        mean_f(weights, return_vector),
        sd_f(weights, covar_table)
    )

In [35]:
def constraint_function(weights):
    # For SciPy equality constraints, we require constraint_function(weights) == 0.
    return np.sum(weights) - 1

In [36]:
def mvu_f(weights):
    # mvu_f is defined for mean-variance optimization (not used below).
    return -(mean_f(weights, return_vector) - 0.5 * gamma * sd_f(weights, covariances)**2)

In [37]:
def r_req_f(goal_vector, goal_allocation, pool):
    return (goal_vector[1] / (goal_allocation * pool))**(1 / goal_vector[2]) - 1

In [38]:
def get_goal_data(master_excel_path, plan=Profile):
    """
    Returns a DataFrame of the specified goal table (P1 or P2).
    P1 => B2:F5
    P2 => B7:F10
    """
    if plan == "P1":
        skip = 1  # start reading at row 2
    elif plan == "P2":
        skip = 6  # start reading at row 7
    else:
        raise ValueError("Plan not recognized. Use 'P1' or 'P2'.")

    df_goals = pd.read_excel(master_excel_path,sheet_name="Goals",skiprows=skip,nrows=4,usecols="B:F",header=0)
    # First column is "Goal Info", so make that the index
    df_goals.set_index(df_goals.columns[0], inplace=True)
    return df_goals



# Load & Parse Data

In [39]:

## -- Repo Root and Folders -- ##

# Get repo root and set folders
root = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True).stdout.strip()
data_folder = os.path.join(root, "GBI Optimisation", "data")
output_folder = os.path.join(root, "GBI Optimisation")

# Get excel file and sheets
master_excel_path = os.path.join(data_folder, "Master.xlsx")

df_returns = pd.read_excel(master_excel_path, sheet_name=excel_returns)
df_vols = pd.read_excel(master_excel_path, sheet_name=excel_volatilities)

df_returns.set_index(df_returns.columns[0], inplace=True)
df_vols.set_index(df_vols.columns[0], inplace=True)

df_corr = pd.read_excel(master_excel_path, sheet_name=excel_correlation)
df_corr.set_index(df_corr.columns[0], inplace=True)


In [40]:
# - NEW V5 CONTENT REGARDING AKTIESPAREKONTO ALLOCATION - #

# Initialize tracker for account allocations
all_account_allocations = []

# Aktiesparekonto cap
aktiesparekonto_cap = 166200
aktiesparekonto_used_total = 0

In [41]:
infeasible_goals_log = [] #Added for the new phi_f definition


while True:
    ## -- Loop Table -- ##
    table_loop_df = pd.read_excel(master_excel_path, sheet_name="Loop",usecols="B:F",skiprows=1,header=0)
    # Find Last looped year and select the next one
    # Filter only rows where LoopStatus is N
    pending_rows = table_loop_df[table_loop_df["LoopStatus"] == "N"]

    # If we find any rows, get the row with the lowest Year
    if pending_rows.empty:
        print("All loops completed")
        break


    chosen_row = pending_rows.loc[pending_rows["Year"].idxmin()]
    loop_year = chosen_row["Year"]
    loop_number = chosen_row["N"]
    loop_age1 = chosen_row["AgeP1"]
    loop_age2 = chosen_row["AgeP2"]

    print(loop_year, loop_number, loop_age1, loop_age2)

    # - Wealth - #

    # Retrieve Wealth value for selected profile and year from "Wealth" sheet
    df_wealth = pd.read_excel(master_excel_path, sheet_name="Wealth")
    df_wealth.set_index(df_wealth.columns[0], inplace=True)  # Set P1/P2 as index

    # Use profile and loop_year to get the correct pool value
    try:
        pool = df_wealth.loc[Profile, str(loop_year)]
    except KeyError:
        raise KeyError(f"Wealth data for profile '{Profile}' in year {loop_year} not found.")

    capital_market_expectations_raw = {}
    for asset in df_returns.index:
        expected_return = df_returns.loc[asset, str(loop_year)]
        volatility = df_vols.loc[asset, 'volatility']
        capital_market_expectations_raw[asset] = {
            'Return Forecast': expected_return,
            'Volatility Forecast': volatility
        }

    capital_market_expectations_raw = pd.DataFrame.from_dict(capital_market_expectations_raw, orient='index')

    capital_market_expectations_raw = capital_market_expectations_raw.reset_index()
    capital_market_expectations_raw.rename(columns={'index': 'Unnamed: 0'}, inplace=True)

    # Rearrange columns to match your old format (optional):
    capital_market_expectations_raw = capital_market_expectations_raw[['Unnamed: 0', 'Return Forecast', 'Volatility Forecast']]

    goal_data_raw = get_goal_data(master_excel_path, plan="P1")

    # - Dynamic time horizon based on current year - #

    # Step 1: Get Starting Year (minimum year in Loop sheet)
    starting_year = table_loop_df["Year"].min()

    # Step 2: Compute Goal Years = starting_year + time_horizon
    goal_horizons = goal_data_raw.loc["Time Horizon"].astype(int)
    goal_years = starting_year + goal_horizons

    # Step 3: Recalculate Time Horizons = goal_years - current loop_year
    adjusted_horizons = goal_years - loop_year

    # Step 4: Replace the "Time Horizon" row in goal_data_raw
    goal_data_raw.loc["Time Horizon"] = adjusted_horizons

    goals = ["A", "B", "C", "D"]
    active_goal_mask = np.array([adjusted_horizons[f"GOAL {g}"] > 0 for g in goals])

    # Optional: print to confirm
    print("Starting Year:", starting_year)
    print("Current Year:", loop_year)
    print("Goal Years:", goal_years.to_dict())
    print("Adjusted Horizons:", adjusted_horizons.to_dict())

    # Record number of potential investments and goals
    num_assets = capital_market_expectations_raw.shape[0]
    num_goals = goal_data_raw.shape[1]

    # Create vector of expected returns
    return_vector = capital_market_expectations_raw["Return Forecast"].to_numpy()

    # Get the correlations as a numeric DataFrame (just a num_assets × num_assets block)
    correlations = df_corr.iloc[:num_assets, :num_assets].astype(float)

    # Build the covariance matrix: stdev_i * stdev_j * correlation_ij
    stdevs = capital_market_expectations_raw["Volatility Forecast"].to_numpy()
    covariances = np.zeros((num_assets, num_assets))
    for i in range(num_assets):
        for j in range(num_assets):
            covariances[i, j] = stdevs[i] * stdevs[j] * correlations.iloc[i, j]

    goal_A = goal_data_raw["GOAL A"].values
    goal_B = goal_data_raw["GOAL B"].values
    goal_C = goal_data_raw["GOAL C"].values
    goal_D = goal_data_raw["GOAL D"].values

    # - Optimal Goal Allocation - #


    goal_allocation = np.arange(0.01, 1.01, 0.01)

    # Starting weights (random initialization normalized to sum to 1)
    starting_weights = np.random.uniform(0, 1, num_assets)
    starting_weights /= np.sum(starting_weights)

    # Initialize matrices to store the optimal weights for each goal
    optimal_weights_A = np.zeros((len(goal_allocation), num_assets))
    optimal_weights_B = np.zeros((len(goal_allocation), num_assets))
    optimal_weights_C = np.zeros((len(goal_allocation), num_assets))
    optimal_weights_D = np.zeros((len(goal_allocation), num_assets))

    goal_allocation = np.arange(0.01, 1.01, 0.01)

    # Starting weights (random initialization normalized to sum to 1)
    starting_weights = np.random.uniform(0, 1, num_assets)
    starting_weights /= np.sum(starting_weights)

    # Initialize matrices to store the optimal weights for each goal
    optimal_weights_A = np.zeros((len(goal_allocation), num_assets))
    optimal_weights_B = np.zeros((len(goal_allocation), num_assets))
    optimal_weights_C = np.zeros((len(goal_allocation), num_assets))
    optimal_weights_D = np.zeros((len(goal_allocation), num_assets))

    # Set SLSQP options to be more stringent, mimicking solnp's behavior.
    slsqp_opts = {
        'ftol': 1e-12,     # function tolerance
        'eps': 1e-12,      # finite-difference step size for gradient estimation
        'maxiter': 10000,  # maximum iterations
        'disp': False     # do not display convergence messages
    }

    for i, alloc in enumerate(goal_allocation):
        allocation = alloc      # Global variable used in optim_function
        covar_table = covariances

        # Goal A Optimization
        goal_vector = goal_A   # Global variable used in optim_function
        if goal_A[1] <= pool * allocation:
            optimal_weights_A[i, :] = [0]*(num_assets - 1) + [1]
        else:
            result = minimize(
                optim_function,
                starting_weights,
                constraints=[{'type': 'eq', 'fun': constraint_function}],
                bounds=[(0, 1)] * num_assets,
                method='SLSQP',
                options=slsqp_opts
            )
            optimal_weights_A[i, :] = result.x

        # Goal B Optimization
        goal_vector = goal_B
        if goal_B[1] <= pool * allocation:
            optimal_weights_B[i, :] = [0]*(num_assets - 1) + [1]
        else:
            result = minimize(
                optim_function,
                starting_weights,
                constraints=[{'type': 'eq', 'fun': constraint_function}],
                bounds=[(0, 1)] * num_assets,
                method='SLSQP',
                options=slsqp_opts
            )
            optimal_weights_B[i, :] = result.x

        # Goal C Optimization
        goal_vector = goal_C
        if goal_C[1] <= pool * allocation:
            optimal_weights_C[i, :] = [0]*(num_assets - 1) + [1]
        else:
            result = minimize(
                optim_function,
                starting_weights,
                constraints=[{'type': 'eq', 'fun': constraint_function}],
                bounds=[(0, 1)] * num_assets,
                method='SLSQP',
                options=slsqp_opts
            )
            optimal_weights_C[i, :] = result.x

        # Goal D Optimization
        goal_vector = goal_D
        if goal_D[1] <= pool * allocation:
            optimal_weights_D[i, :] = [0]*(num_assets - 1) + [1]
        else:
            result = minimize(
                optim_function,
                starting_weights,
                constraints=[{'type': 'eq', 'fun': constraint_function}],
                bounds=[(0, 1)] * num_assets,
                method='SLSQP',
                options=slsqp_opts
            )
            optimal_weights_D[i, :] = result.x

    # Calculate the best probability (phi) for each allocation level for every goal
    phi_A = np.zeros(len(goal_allocation))
    phi_B = np.zeros(len(goal_allocation))
    phi_C = np.zeros(len(goal_allocation))
    phi_D = np.zeros(len(goal_allocation))

    for i, alloc in enumerate(goal_allocation):
        phi_A[i] = phi_f(goal_A, alloc, pool,
                         mean_f(optimal_weights_A[i, :], return_vector),
                         sd_f(optimal_weights_A[i, :], covariances), year=loop_year, goal_name="Goal A")
        phi_B[i] = phi_f(goal_B, alloc, pool,
                         mean_f(optimal_weights_B[i, :], return_vector),
                         sd_f(optimal_weights_B[i, :], covariances), year=loop_year, goal_name="Goal B")
        phi_C[i] = phi_f(goal_C, alloc, pool,
                         mean_f(optimal_weights_C[i, :], return_vector),
                         sd_f(optimal_weights_C[i, :], covariances), year=loop_year, goal_name="Goal C")
        phi_D[i] = phi_f(goal_D, alloc, pool,
                         mean_f(optimal_weights_D[i, :], return_vector),
                         sd_f(optimal_weights_D[i, :], covariances), year=loop_year, goal_name="Goal D")

    # Simulate goal weights: each row is a simulated allocation (in percentages)
    sim_goal_weights = np.random.multinomial(100, [1/num_goals]*num_goals, size=n_trials) #this one sums to 100 so its good
    for i in range(n_trials):
        rand_vector = np.random.uniform(0, 1, num_goals)
        normalizer = np.sum(rand_vector)
        percents = np.round((rand_vector / normalizer) * 100, 0)

        # Only enforce floor for active goals
        floor_applied = np.where(active_goal_mask, np.maximum(percents, 1), percents)
        sim_goal_weights[i, :] = floor_applied


    # Calculate utility for each simulated portfolio.
    # Note: subtract 1 from simulated weights for 0-indexing.
    utility = (
        goal_A[0] * phi_A[sim_goal_weights[:, 0] - 1] +
        goal_A[0] * goal_B[0] * phi_B[sim_goal_weights[:, 1] - 1] +
        goal_A[0] * goal_B[0] * goal_C[0] * phi_C[sim_goal_weights[:, 2] - 1] +
        goal_A[0] * goal_B[0] * goal_C[0] * goal_D[0] * phi_D[sim_goal_weights[:, 3] - 1]
    )

    # Find the index of the portfolio with the highest utility
    index = np.argmax(utility)
    optimal_goal_weights = sim_goal_weights[index, :]

    # - Optimal Subportfolio Allocation - #

    # Retrieve optimal subportfolio allocations
    optimal_subportfolios = np.zeros((num_goals, num_assets))

    # For each goal, use the simulated percentage to select the corresponding row
    # from the optimal weights matrix (adjust for zero-indexing)
    for i in range(num_goals):
        optimal_subportfolios[i, :] = eval(f"optimal_weights_{goals[i]}")[optimal_goal_weights[i] - 1, :]

    # Compute the optimal aggregate investment portfolio.
    optimal_aggregate_portfolio = (optimal_goal_weights / 100) @ optimal_subportfolios

    #Define asset names
    asset_names = capital_market_expectations_raw.iloc[:, 0].astype(str).tolist()

    # - Storing and exporting results - #

    # Create a DataFrame for the aggregate portfolio.
    # First calculate unrounded percentages
    raw_alloc = optimal_aggregate_portfolio * 100

    # Normalize to force sum = 100 after rounding
    normalized_alloc = raw_alloc / raw_alloc.sum() * 100
    normalized_goal_alloc = np.zeros_like(optimal_goal_weights, dtype=float)
    active_sum = np.sum(optimal_goal_weights[active_goal_mask])
    normalized_goal_alloc[active_goal_mask] = (optimal_goal_weights[active_goal_mask] / active_sum) * 100

    # Create a DataFrame for the across-goal allocation.
    df_across_goal = pd.DataFrame({
        "Goal": goals,
        "Allocation (%)": np.round(normalized_goal_alloc, 2) # Keep raw percentages
    })

    # Round after normalization
    df_aggregate = pd.DataFrame({
        "Asset": asset_names,
        "Weight": optimal_aggregate_portfolio,
        "Allocation (%)": np.round(normalized_alloc, 2)
    })

    # - NEW V5 CONTENT REGARDING AKTIESPAREKONTO ALLOCATION - #
    account_allocations = []

    for i, row in df_aggregate.iterrows():
        asset_name = row["Asset"]
        weight = row["Weight"]
        allocation_value = weight * pool  # actual money invested this year

        if "Equities" in asset_name:
            remaining_cap = aktiesparekonto_cap - aktiesparekonto_used_total
            if remaining_cap > 0:
                invested_in_ask = min(allocation_value, remaining_cap)
                invested_in_normal = allocation_value - invested_in_ask
                aktiesparekonto_used_total += invested_in_ask
            else:
                invested_in_ask = 0
                invested_in_normal = allocation_value
        else:
            invested_in_ask = 0
            invested_in_normal = allocation_value

        account_allocations.append({
            "Year": loop_year,
            "Asset": asset_name,
            "ASK (DKK)": invested_in_ask,
            "Normal (DKK)": invested_in_normal
        })

    df_accounts = pd.DataFrame(account_allocations)
    all_account_allocations.append(df_accounts)

    # - V5 END - #

    print("Optimal Across-Goal Allocation:")
    print(df_across_goal.to_string(index=False))

    print("\nOptimal Aggregate Investment Allocation:")
    print(df_aggregate.to_string(index=False))

    # -- Safer Excel launch --
    app = xw.App(visible=False)
    app.display_alerts = False
    app.screen_updating = False

    time.sleep(1)

    # Open workbook
    wb = app.books.open(master_excel_path)
    ws = wb.sheets[excel_gbi]

    # Get header values from row 1 (columns B to AY ~= cols 2 to 51)
    header_values = [ws.cells(1, col).value for col in range(2, 53)]

    try:
        year_col = header_values.index(str(loop_year)) + 2
    except ValueError:
        raise ValueError(f"Year {loop_year} not found in worksheet headers.")

    # Write weights
    for i, asset in enumerate(asset_names):
        allocation = float(np.round(normalized_alloc[i], 2)) / 100
        cell = ws.cells(i + 2, year_col)
        cell.value = allocation
        cell.number_format = '0.00%'  # display as percentage

    # -- Export Goal Weights to Excel -- #

    # Re-access the sheet after workbook is open
    ws_goals = wb.sheets[excel_gbi_goals]

    # Read header values from row 1 (columns B to AY ≈ cols 2 to 52)
    goal_header_values = [ws_goals.cells(1, col).value for col in range(2, 53)]

    try:
        goal_year_col = goal_header_values.index(str(loop_year)) + 2
    except ValueError:
        raise ValueError(f"Year {loop_year} not found in goal worksheet headers.")

    # Write each across-goal allocation to the appropriate row
    for i, allocation in enumerate(df_across_goal["Allocation (%)"]):
        value = float(np.round(allocation / 100, 6))  # convert to decimal
        row = i + 2  # assuming goal names are in rows starting at 2
        cell = ws_goals.cells(row, goal_year_col)
        cell.value = value
        cell.number_format = '0.00%'  # display as percentage

    # -- Modify Loop Check -- #
    print("Starting loop status check...")

    ws_loop = wb.sheets["Loop"]
    print("Accessed 'Loop' worksheet.")

    # Read data range again (columns B to F)
    last_row = ws_loop.cells.last_cell.row
    print(f"Last cell row: {last_row}")

    loop_data = ws_loop.range("B2:F" + str(last_row)).value
    print(f"Loaded loop data. Total rows read: {len(loop_data)}")

    # Find and update the matching year with LoopStatus 'N'
    found = False
    for i, row in enumerate(loop_data):
        year = row[1]
        status = row[4]
        print(f"Row {i+2}: Year = {year}, Status = {status}")
        if year == loop_year and status == 'N':
            print(f"Match found at row {i+2}. Updating status to 'Y'.")
            ws_loop.cells(i + 2, 6).value = 'Y'  # Column F is column 6
            found = True
            break

    if not found:
        print(f"No matching row found for year {loop_year} with status 'N'.")
    else:
        print("Status updated successfully.")

    wb.save()
    wb.close()
    app.quit()
    time.sleep(1)
    table_loop_df = pd.read_excel(master_excel_path, sheet_name="Loop", usecols="B:F", skiprows=1, header=0)


2025 1 25 40
Starting Year: 2025
Current Year: 2025
Goal Years: {'GOAL A': 2035, 'GOAL B': 2040, 'GOAL C': 2045, 'GOAL D': 2070}
Adjusted Horizons: {'GOAL A': 10, 'GOAL B': 15, 'GOAL C': 20, 'GOAL D': 45}
Optimal Across-Goal Allocation:
Goal  Allocation (%)
   A            0.98
   B           93.14
   C            3.92
   D            1.96

Optimal Aggregate Investment Allocation:
                               Asset   Weight  Allocation (%)
         Emerging Markets - Equities 0.532525           52.21
        Developed Markets - Equities 0.064480            6.32
Emerging Markets State - Obligations 0.006395            0.63
      High Yield Bonds - Obligations 0.005803            0.57
Investment Grade Bonds - Obligations 0.407087           39.91
   Government ZC Bonds - Obligations 0.003710            0.36
Starting loop status check...
Accessed 'Loop' worksheet.
Last cell row: 1048576
Loaded loop data. Total rows read: 1048575
Row 2: Year = Year, Status = LoopStatus
Row 3: Year = 2025.

In [42]:
# - NEW V5 CONTENT REGARDING AKTIESPAREKONTO ALLOCATION - #
final_accounts_df = pd.concat(all_account_allocations, ignore_index=True)
final_accounts_df.to_csv(os.path.join(output_folder, "account_allocations_log.csv"), index=False)

In [43]:
if infeasible_goals_log:
    print("\n⚠️ Infeasible goal scenarios encountered:")
    for year, goal, reason in infeasible_goals_log:
        print(f" - Year {year}, Goal {goal}: {reason}")

    # Optional: write to CSV
    pd.DataFrame(infeasible_goals_log, columns=["Year", "Goal", "Reason"]).to_csv(
        os.path.join(output_folder, "infeasible_goals_log.csv"), index=False)
else:
    print("\n✅ No infeasible goals encountered.")



⚠️ Infeasible goal scenarios encountered:
 - Year 2028, Goal Goal A: Required return > 100% (109.68%)
